In [92]:
"""
====================================================
Download ERA5 Data
====================================================
"""

'\n====================================================\nDownload ERA5 Data\n====================================================\n'

In [93]:
#######################
# DIRECTORIES

In [94]:
#SETTING UP DIRECOTRIES
mainDirectory = "/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/"
dataDirectory=mainDirectory+"../DATA/ERA5_Data/"
import os; os.makedirs(dataDirectory, exist_ok=True)

In [95]:
#######################
# LIBRARIES, FUNCTIONS, and CLASSES

In [96]:
# IMPORT LIBRARIES
# --- Add your Functions folder to sys.path ---
import sys

path = mainDirectory + "/Libraries/"
sys.path.append(path)

# --- Import all your function modules ---
import importlib

modules = [
    "Libraries",
]

for mod in modules:
    globals()[mod] = importlib.import_module(mod)  # import module itself
    globals().update(vars(globals()[mod]))  # import all functions into global namespace

In [97]:
# IMPORT FUNCTIONS
# --- Add your Functions folder to sys.path ---
import sys

path = mainDirectory + "Functions_2.0/"
sys.path.append(path)


# --- Import all your function modules ---
import importlib

modules = [
    "AreaAverageFunctions",
    "ComputationFunctions",
    "DataFunctions",
    "DerivativeFunctions",
    "PlottingFunctions",
    "StatisticalFunctions",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)  # import module itself
    globals().update(vars(globals()[mod]))  # import all functions into global namespace

In [98]:
# IMPORT CLASSES
# --- Add your Functions folder to sys.path ---
import sys

path = mainDirectory + "Functions_2.0/Classes/"
sys.path.append(path)


# --- Import all your function modules ---
import importlib

modules = [
    "Classes_1",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)  # import module itself
    globals().update(vars(globals()[mod]))  # import all functions into global namespace

In [99]:
###########################
# DOWNLOADING DATA FUNCTIONS

In [103]:
# DOWNLOADING ERA5
# Code Inspired from "Download_ERA5_with_python" by github.com/joaohenry23 at https://github.com/joaohenry23/Download_ERA5_with_python

#PRELIMINARY STEPS
#(1) go to https://cds.climate.copernicus.eu/how-to-api
#(2) make file in main user directory called .cdsapirc
#(3) copy the following into file: 
#    url: https://cds.climate.copernicus.eu/api
#    key: 6d55399f-dbc5-48bd-848c-31168fc0b133 (key will be different for you based on your account)
#(4) pip install "cdsapi>=0.7.4"

import cdsapi 
def DownloadERA5(variables, date, area):
    c = cdsapi.Client()
    for variable in tqdm(variables, desc="Downloading ERA5 variables"):
        print(f"Downloading {variable}", "\n")
        c.retrieve(
            "reanalysis-era5-pressure-levels",
            {
                "product_type": "reanalysis",
                "format": "netcdf",
                "variable": variable,
                # "pressure_level": ['100', '250', '500', '750', '1000'], #LOW-RES
                "pressure_level": [
                    '10', '20', '30', '50', '70', 
                    '100', '125', '150', '175', '200', '225',
                    '250', '300', '350', '400', '450', '500',
                    '550', '600', '650', '700', '750', '775',
                    '800', '825', '850', '875', '900', '925',
                    '950', '975', '1000',
                ]

                "date": date,
                "time": [f"{h:02d}:00" for h in range(24)],
                "area": area,
                "grid": [0.25, 0.25],
            },
            os.path.join(
                dataDirectory, date_folder, f"{variable}_ERA5_{date_folder}.nc"
            ),
        )

#EXAMPLE RUN
# date_string = "06-30 - 07-02 (2022)"
# date_folder = MakeDateFolder(date_string)
# date_string_converted = date_string_to_range(date_string)

# # running
# DownloadERA5_V2(variables,date_string_converted, area)

In [104]:
# DATE INFORMATION
def date_string_to_range(date_string: str) -> str:
    """
    Convert a date string like "06-30 - 07-02 (2022)"
    into ERA5 API format: "2022-06-30/to/2022-07-02".
    """
    # Extract year
    year = date_string.split("(")[1].replace(")", "").strip()

    # Extract the two parts safely
    date_part = date_string.split("(")[0].strip()  # "06-30 - 07-02"
    start, end = date_part.split(" - ")            # ["06-30", "07-02"]

    # Make full YYYY-MM-DD
    start_date = f"{year}-{start}"
    end_date   = f"{year}-{end}"

    return f"{start_date}/{end_date}"

    
def MakeDateFolder(date_string):
    date_folder = strings.DateString(date_string)
    # adding date to output folder
    subdir = os.path.join(dataDirectory, date_folder)
    os.makedirs(subdir, exist_ok=True)
    return date_folder


# COORDINATES INFORMATION
def GetCoordinates(longitude, latitude, dx_m=250e3, dy_m=250e3, grid_res=0.25):
    longitude = coordinates.DMSToDecimal(*longitude)
    latitude = coordinates.DMSToDecimal(*latitude)

    dlon = coordinates.dxTOdlon(dx_m=dx_m, lat_deg=latitude)
    dlat = coordinates.dyTOdlat(dy_m=dy_m)

    N, W, S, E = [latitude + dlat, longitude - dlon, latitude - dlat, longitude + dlon]
    print("Coords box:", [N, W, S, E])
    # Round outward to 0.25 grid
    N = math.ceil(N / grid_res) * grid_res  # round north up
    S = math.floor(S / grid_res) * grid_res  # round south down
    W = math.floor(W / grid_res) * grid_res  # round west down (more negative)
    E = math.ceil(E / grid_res) * grid_res  # round east up

    area = [N, W, S, E]
    print("Rounded box:", area)
    return area


# VARIABLES INFORMATION
def GetVariableNames():
    variables = [
        "u_component_of_wind",
        "v_component_of_wind",
        "vertical_velocity",
        "divergence",
        "vorticity",
        "temperature",
        "specific_humidity",
        "specific_cloud_liquid_water_content",
        "specific_cloud_ice_water_content",
        "specific_rain_water_content",
        "relative_humidity",
        "cloud_cover",
        "geopotential",
    ]
    return variables

In [105]:
###########################
# DOWNLOADING TRACER DATA
#resolution: 72*32*20*23 = 1059840 grid-points

In [106]:
# coorindates information
# GETTING BOUNDING BOX centered at Houston, TX Mobile Facility (TRACER) Facility S2 ==> CSAP (C-Band Scanning ARM Precipitation Radar)
# 29°31'55"N, 95°17'2"W
longitude = (95, 17, 2, "W")
latitude = (29, 31, 55, "N")
area = GetCoordinates(longitude, latitude)
variables = GetVariableNames()

Coords box: [31.78024845924127, -97.86790574593715, 27.28364042964762, -92.69987203184061]
Rounded box: [32.0, -98.0, 27.25, -92.5]


In [84]:
###########################
# DATE ONE (BORING CASE)

In [107]:
# INFORMATION
# date information
date_string = "06-08 - 06-10 (2022)"
date_folder = MakeDateFolder(date_string)
date_string_converted = date_string_to_range(date_string)

# running
DownloadERA5(variables,date_string_converted, area)

2025-09-04 14:16:02,912 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.

2025-09-04 14:16:03,427 INFO Request ID is 1c6e65d1-405e-4fe1-aa60-81d41c62cdfa
2025-09-04 14:16:03,606 INFO status has been updated to accepted
2025-09-04 14:16:12,428 INFO status has been updated to running
2025-09-04 14:20:24,646 INFO status has been updated to successful


7f6eba9df90b14ee8293d326a5e88410.nc:   0%|          | 0.00/2.19M [00:00<?, ?B/s]

2025-09-04 14:20:27,288 INFO Request ID is e9e33ff1-5fcd-47d9-ae0b-a812fb03800b
2025-09-04 14:20:27,442 INFO status has been updated to accepted
2025-09-04 14:20:36,239 INFO status has been updated to running
2025-09-04 14:24:47,937 INFO status has been updated to successful


603bfcb62ac8b778cdceb0182c3e2e90.nc:   0%|          | 0.00/2.24M [00:00<?, ?B/s]

...

372e22d4fe628b9d46f459fb13b14cd0.nc:   0%|          | 0.00/2.03M [00:00<?, ?B/s]

2025-09-04 15:02:57,951 INFO Request ID is 9463b50a-5a6e-4b7d-982e-53393407dcb3
2025-09-04 15:02:58,123 INFO status has been updated to accepted
2025-09-04 15:03:12,907 INFO status has been updated to running
2025-09-04 15:07:19,727 INFO status has been updated to successful


1faf0fd18187601992043c0bea58e48b.nc:   0%|          | 0.00/300k [00:00<?, ?B/s]

2025-09-04 15:07:22,019 INFO Request ID is ae4f4303-4264-4ea2-8b37-9b7eadf4bfd9
2025-09-04 15:07:22,191 INFO status has been updated to accepted
2025-09-04 15:07:31,018 INFO status has been updated to running
2025-09-04 15:11:42,639 INFO status has been updated to successful


bf6961ad7382d959315951c2365c6592.nc:   0%|          | 0.00/1.45M [00:00<?, ?B/s]

In [108]:
###########################
# DATE TWO (RAINY CASE)

In [109]:
# INFORMATION
# date information
date_string = "06-30 - 07-02 (2022)"
date_folder = MakeDateFolder(date_string)
date_string_converted = date_string_to_range(date_string)

# running
DownloadERA5(variables,date_string_converted, area)

2025-09-04 15:11:45,617 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.

2025-09-04 15:11:45,983 INFO Request ID is 72b542fa-1dfe-4e40-8f11-813a5b006272
2025-09-04 15:11:46,173 INFO status has been updated to accepted
2025-09-04 15:11:55,013 INFO status has been updated to running
2025-09-04 15:16:06,660 INFO status has been updated to successful


7b4a4e2026fa7c690070f716ddd43be5.nc:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

2025-09-04 15:16:09,499 INFO Request ID is 916bbef7-3731-40d4-af59-0cb699ded33b
2025-09-04 15:16:09,773 INFO status has been updated to accepted
2025-09-04 15:16:18,657 INFO status has been updated to running
2025-09-04 15:20:30,219 INFO status has been updated to successful


1def63c59368cb35bba1cc3339eade8a.nc:   0%|          | 0.00/2.32M [00:00<?, ?B/s]

...

24baa8ac92f7cf0fed74bd0a0cff2d53.nc:   0%|          | 0.00/2.01M [00:00<?, ?B/s]

2025-09-04 15:58:38,448 INFO Request ID is 97e72652-1093-407a-b210-499925e59146
2025-09-04 15:58:38,622 INFO status has been updated to accepted
2025-09-04 15:58:48,065 INFO status has been updated to running
2025-09-04 16:03:00,185 INFO status has been updated to successful


6488e204beca1a775abaee6fff1ffe16.nc:   0%|          | 0.00/576k [00:00<?, ?B/s]

2025-09-04 16:03:03,442 INFO Request ID is 12316073-a986-49e3-b4ae-04f486bd81f7
2025-09-04 16:03:03,633 INFO status has been updated to accepted
2025-09-04 16:03:17,660 INFO status has been updated to running
2025-09-04 16:07:28,586 INFO status has been updated to successful


bcab6ce775d7575f49499f8366a3493c.nc:   0%|          | 0.00/1.46M [00:00<?, ?B/s]

In [110]:
###########################
# DATE THREE (INTERESTING CASE)

In [111]:
# INFORMATION
# date information
date_string = "08-11 - 08-13 (2022)"
date_folder = MakeDateFolder(date_string)
date_string_converted = date_string_to_range(date_string)

# running
DownloadERA5(variables,date_string_converted, area)

2025-09-04 16:07:31,429 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.

2025-09-04 16:07:31,800 INFO Request ID is 3f897076-4316-4523-9d0a-491b2bf6ba49
2025-09-04 16:07:31,955 INFO status has been updated to accepted
2025-09-04 16:07:46,174 INFO status has been updated to running
2025-09-04 16:11:52,458 INFO status has been updated to successful


3fd8fa705a1abeb9b41891d9b1abe713.nc:   0%|          | 0.00/2.27M [00:00<?, ?B/s]

2025-09-04 16:11:55,089 INFO Request ID is 756e7511-e1d2-4a32-9d12-ae2e28070925
2025-09-04 16:11:55,257 INFO status has been updated to accepted
2025-09-04 16:12:09,310 INFO status has been updated to running
2025-09-04 16:16:15,472 INFO status has been updated to successful


9776c748ca99a0785c86008531a6b246.nc:   0%|          | 0.00/2.28M [00:00<?, ?B/s]

...

3927a1ccabbb78f0410847c038b389cf.nc:   0%|          | 0.00/1.95M [00:00<?, ?B/s]

2025-09-04 16:54:24,554 INFO Request ID is bce5bd7c-08a5-455d-bd0c-c7307c90d7e9
2025-09-04 16:54:24,727 INFO status has been updated to accepted
2025-09-04 16:54:38,934 INFO status has been updated to running
2025-09-04 16:58:45,714 INFO status has been updated to successful


5d6f9ff51f9ead0507f628d9523e98a1.nc:   0%|          | 0.00/604k [00:00<?, ?B/s]

2025-09-04 16:58:50,220 INFO Request ID is 164795fc-ccb8-4683-bfd0-d81cf7f9e5f4
2025-09-04 16:58:50,415 INFO status has been updated to accepted
2025-09-04 16:58:59,235 INFO status has been updated to running
2025-09-04 17:03:10,833 INFO status has been updated to successful


bfd937a3f75a06aae21b0eb5da9e0c78.nc:   0%|          | 0.00/1.45M [00:00<?, ?B/s]